## KNN usando el dataset Iris

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import Normalizer
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

from collections import Counter

### EDA (Exploratory Data Analysis) en el dataset Iris

Atributos:

1. sepal length en cm
2. sepal width en cm
3. petal length en cm
4. petal width en cm

Utilizaremos solo dos características (features) para facilitar la visualización, sepal length y sepal width.

Clases:

- Iris Setosa
- Iris Versicolour
- Iris Virginica

### Cargamos el Dataset

In [ ]:
# importamos el dataset iris
iris = datasets.load_iris()
# np.c_ es la función para concatenar en numpy
iris_df = pd.DataFrame(data= np.c_[iris['data'], iris['target']],
                      columns= iris['feature_names'] + ['target'])
iris_df.tail()

In [ ]:
iris_df.describe()

### Separamos el dataset en X y Y

In [ ]:
# Una opción para separar las columnas en X y Y, es usando slicing
x = iris_df.iloc[:, :-1]
y = iris_df.iloc[:, -1]

In [ ]:
# Otra opción para separar las columnas en X y Y
#sepal length (cm)	sepal width (cm)	petal length (cm)	petal width (cm)
atributos = ['sepal length (cm)','sepal width (cm)','petal length (cm)','petal width (cm)']
x = iris_df[atributos]
y = iris_df.target #y = iris_df['target']

In [ ]:
x.head()

In [ ]:
y.tail()

In [ ]:
x.head()

In [ ]:
y.head()

### Dividimos el dataset en conjuntos de entrenamiento (training) y prueba (test)

In [ ]:
# dividimos los datos en entrenamiento y prueba
x_train, x_test, y_train, y_test= train_test_split(x, y,
                                                   test_size= 0.2,
                                                   shuffle= True, # revolvemos los datos para evitar sesgo (bias)
                                                   random_state= 0)
x_train= np.asarray(x_train)
y_train= np.asarray(y_train)

x_test= np.asarray(x_test)
y_test= np.asarray(y_test)

In [ ]:
print(f'Tamaño del dataset de entrenamiento: {x_train.shape[0]} ejemplos (samples) \nTamaño del dataset de prueba: {x_test.shape[0]} ejemplos (samples)')

### Normalizamos los datos

In [ ]:
scaler= Normalizer().fit(x_train) # El escalador se ajusta al conjunto de entrenamiento
normalized_x_train= scaler.transform(x_train) # el escalador es aplicado al conjunto de entrenamiento
normalized_x_test= scaler.transform(x_test) # el escalador es aplicado al conjunto de prueba

In [ ]:
print('x train antes de la normalización')
print(x_train[0:5])
print('\nx train después de la normalización')
print(normalized_x_train[0:5])

### Visualizamos los datos antes y después de la Normalización

In [ ]:
## Antes
# Ver las relaciones entre variables; código de color con base al tipo de especie
di= {0.0: 'Setosa', 1.0: 'Versicolor', 2.0:'Virginica'} # dictionary

before= sns.pairplot(iris_df.replace({'target': di}), hue= 'target')
before.figure.suptitle('Pair Plot of the dataset Before normalization', y=1.08)

## Después
iris_df_2= pd.DataFrame(data= np.c_[normalized_x_train, y_train],
                        columns= iris['feature_names'] + ['target'])
di= {0.0: 'Setosa', 1.0: 'Versicolor', 2.0: 'Virginica'}
after= sns.pairplot(iris_df_2.replace({'target':di}), hue= 'target')
after.figure.suptitle('Pair Plot of the dataset After normalization', y=1.08)

### KNN Paso 1 Distancia Euclidiana

In [ ]:
def distance_ecu(x_train, x_test_point):
  """
  Entrada:
    - x_train: correspondiente a los datos de entrenamiento
    - x_test_point: correspondiente al punto de prueba

  Salida:
    -distances: las distancias entre el punto de prueba y cada punto en el dataset de entrenamiento.

  """
  distances= []  ## crear una lista vacía llamada distances
  for row in range(len(x_train)): ## Ciclo en los registros dex_train
      current_train_point= x_train[row] #Se obtienen punto por punto
      current_distance= 0 ## Inicializar la distancia en cero

      for col in range(len(current_train_point)): ## Iteración por las columnas de cada registro

          current_distance += (current_train_point[col] - x_test_point[col]) **2
          ## O current_distance = current_distance + (x_train[i] - x_test_point[i])**2
      current_distance= np.sqrt(current_distance)

      distances.append(current_distance) ## se agrega la distancia a la lista

  # Se almacenan las distancias en un dataframe
  distances= pd.DataFrame(data=distances,columns=['dist'])
  return distances


### KNN Paso 2 Encontrar los K vecinos más cercanos

In [ ]:
def nearest_neighbors(distance_point, K):
    """
    Entrada:
        -distance_point: las distancias entre el punto de prueba y todos los puntos del dataset de entrenamiento.
        -K             : el número de vecinos

    Salida:
        -df_nearest: los k vecinos más cercanos entre el punto de prueba y el conjunto de entrenamiento.

    """

    # Ordena los valores usando la función sort_values
    df_nearest= distance_point.sort_values(by=['dist'], axis=0)

    ## Toma solamente los primeros K neighbors
    df_nearest= df_nearest[:K]
    return df_nearest

### KNN Paso 3 Clasificar el punto con base a voto mayoritario

In [ ]:
def voting(df_nearest, y_train):
    """
    Entrada:
        -df_nearest: dataframe conteniendo los k vecinos más cercanos entre el punto de prueba y el conjunto de entrenamiento.
        -y_train: las etiquetas del conjunto de entrenamiento.

    Salida:
        -y_pred: las predicciones basadas en votación mayoritaria

    """

    ## Usamos el objeto Counter para obtener  the labels with K nearest neighbors.
    counter_vote= Counter(y_train[df_nearest.index])

    y_pred= counter_vote.most_common()[0][0]   # Votación mayoritaria

    return y_pred

### KNN Algoritmo completo

In [ ]:
def KNN_from_scratch(x_train, y_train, x_test, K):

    """
    Entrada:
    -x_train: el dataset completo de entrenamiento
    -y_train: las etiquetas del dataset de entrenamiento
    -x_test: el dataset completo de prueba
    -K: el número de vecinos

    Salida:
    -y_pred: la predicción para todo el conjunto de prueba basada en votación mayoritaria.

    """

    y_pred=[]

    ## Ciclo en el conjunto de prueba para ejecutar los tres pasos
    for x_test_point in x_test:
      distance_point  = distance_ecu(x_train, x_test_point)  ## Paso 1
      df_nearest_point= nearest_neighbors(distance_point, K)  ## Paso 2
      y_pred_point    = voting(df_nearest_point, y_train) ## Paso 3
      y_pred.append(y_pred_point)

    return y_pred


### Probamos el algoritmo KNN en el dataset de prueba

In [ ]:
K=3
y_pred_scratch= KNN_from_scratch(normalized_x_train, y_train, normalized_x_test, K)
print(y_pred_scratch)

### Comparamos nuestra implementación contra la biblioteca Sklearn

In [ ]:
knn=KNeighborsClassifier(K)
knn.fit(normalized_x_train, y_train)
y_pred_sklearn= knn.predict(normalized_x_test)
print(y_pred_sklearn)

### Verificamos si la salida es igual para ambos casos

In [ ]:
print(np.array_equal(y_pred_sklearn, y_pred_scratch))

### Calculamos la exactitud (accuracy) en ambos métodos

In [ ]:
print(f'La exactitud (accuracy) de nuestra implementación es: {accuracy_score(y_test, y_pred_scratch)}')
print(f'La exactitud (accuracy) de la implementation con sklearn es: {accuracy_score(y_test, y_pred_sklearn)}')

### Realizamos ajuste de hiperparámetros (Hyper-parameter Tuning) mediante validación cruzada K-fold

In [ ]:
n_splits= 5 ## Número de partes
kf= KFold(n_splits= n_splits) ## llamamos a la función K Fold

accuracy_k= [] ## Mantenemos registro de las exactitud (accuracy) para cada k
k_values= list(range(1,30,2)) ## Buscar el mejor valor de k

for k in k_values: ## Iteramos en los valores de k
  accuracy_fold= 0
  for normalized_x_train_fold_idx, normalized_x_valid_fold_idx in  kf.split(normalized_x_train): ## Ciclo en las particiones
      normalized_x_train_fold= normalized_x_train[normalized_x_train_fold_idx] ## alimneta los valores
      y_train_fold= y_train[normalized_x_train_fold_idx]

      normalized_x_test_fold= normalized_x_train[normalized_x_valid_fold_idx]
      y_valid_fold= y_train[normalized_x_valid_fold_idx]
      y_pred_fold= KNN_from_scratch(normalized_x_train_fold, y_train_fold, normalized_x_test_fold, k)

      accuracy_fold+= accuracy_score (y_pred_fold, y_valid_fold) ## Acumula la  exactitud (accuracy)
  accuracy_fold= accuracy_fold/ n_splits ## La divide entre el número de particiones
  accuracy_k.append(accuracy_fold)


In [ ]:
print(f'La exactitud (accuracy) para cada valor de K value fue {list ( zip (accuracy_k, k_values))}') ## crea una tupla con la exactitud (accuracy) correspondiente a cada valor de k

In [ ]:
print(f'La mejor exactitud (accuracy) fue: {np.max(accuracy_k)}, la cual corresponde al valor de K= {k_values[np.argmax(accuracy_k)]}')

### Referencias
- https://deepnote.com/app/ndungu/Implementing-KNN-Algorithm-on-the-Iris-Dataset-e7c16493-500c-4248-be54-9389de603f16
